# 03 · Normalization

Phase 3. Turns the pixel keypoints from `02_detect_pose` into a canonical skeleton per fighter and writes
`data/runs/<video_stem>/normalized.npz` (contract in `sf6mr/runs.py`):

1. joints below the confidence threshold are dropped,
2. short left/right limb swaps by RTMPose are undone,
3. gaps of up to 10 frames are interpolated (marked as not observed),
4. light Savitzky-Golay smoothing,
5. **facing** from the two fighters' positions, turning only while grounded (like SF6),
6. hip-centered, divided by torso length, y up, and fighters facing left are mirrored with left/right swapped:
   **every skeleton faces +x**.

No models here. Run `02_detect_pose` on the video first, then *Restart & Run All*.

In [ ]:
# Parameters
CONFIG = "configs/default.yaml"                               # repo-relative
VIDEO = "data/raw/goto-ken_vs_haitani-chunli_0330-0350.mp4"  # same video as 02_detect_pose
OVERRIDES = []              # e.g. ["normalize.max_gap=5", "normalize.smooth_window=0"]
RENDER_VIDEO = True         # write normalized.mp4 (original + both normalized skeletons)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from sf6mr import frames as fr
from sf6mr import runs, visualize
from sf6mr.config import load_config, resolve_path
from sf6mr.normalize import NormalizeParams, normalize_sequence

cfg = load_config(resolve_path(CONFIG), OVERRIDES)
video = resolve_path(VIDEO)
out_dir = runs.run_dir(cfg, video, create=False)
for name in ("frames.json", "keypoints.npz"):
    if not (out_dir / name).exists():
        raise FileNotFoundError(f"{out_dir / name} is missing: run 02_detect_pose on {video.name} first")

meta = runs.load_frames_json(out_dir / "frames.json")
keypoints = runs.load_keypoints(out_dir / "keypoints.npz")
T = len(keypoints)
assert T == meta["frame_count"], "keypoints.npz and frames.json disagree on the frame count"
print(f"{video.name}: {T} frames, {meta['width']}x{meta['height']}")

## Normalize

In [ ]:
params = NormalizeParams.from_config(cfg.get("normalize"))
result = normalize_sequence(keypoints, meta["height"], params)
runs.save_normalized(out_dir / "normalized.npz", result, params)

joints, facing = result["joints"], result["facing"]
before = keypoints[..., 2] >= params.min_score
after = ~np.isnan(joints[..., 0])
for p in range(2):
    switches = (np.flatnonzero(np.diff(facing[:, p]) != 0) + 1).tolist()
    print(f"P{p + 1}: joints usable {before[:, p].mean():.0%} -> {after[:, p].mean():.0%} after interpolation · "
          f"left/right limb swaps undone: {result['lr_fixes'][p]} · grounded {result['grounded'][:, p].mean():.0%} · "
          f"turns around at frames {switches}")
print("wrote", out_dir / "normalized.npz")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
colors = ["#d6337f", "#1f7fe0"]
t = np.arange(T)
for p in range(2):
    axes[0].plot(t, result["center"][:, p, 0], color=colors[p], lw=1, label=f"P{p + 1}")
    axes[1].step(t, facing[:, p] + (0.06 if p else -0.06), where="post", color=colors[p], lw=1.5, label=f"P{p + 1}")
    axes[2].plot(t, result["scale"][:, p], color=colors[p], lw=1, label=f"P{p + 1}")
axes[0].set(ylabel="hip x (px)", title="Position, facing and scale")
axes[1].set(ylabel="facing", yticks=[-1, 0, 1], yticklabels=["left", "?", "right"])
axes[2].set(ylabel="torso (px)", xlabel="frame", xlim=(0, T))
for ax in axes:
    ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## Check: normalized skeletons

Both fighters should face right in every panel (x = forward), whichever side of the screen they are on.
Hollow joints are interpolated. Left limbs are drawn lighter.

In [ ]:
show = np.linspace(30, T - 30, 6).astype(int)
fig, axes = plt.subplots(2, len(show), figsize=(2.6 * len(show), 6))
for c, t_ in enumerate(show):
    for p in range(2):
        arrow = {1: "->", -1: "<-", 0: "?"}[int(facing[t_, p])]  # OpenCV fonts are ASCII-only
        panel = visualize.draw_normalized(
            joints[t_, p], visualize.PLAYER_COLORS[p], size=(320, 400), px_per_unit=80,
            observed=result["observed"][t_, p], title=f"P{p + 1} {t_} {arrow}",
        )
        axes[p, c].imshow(cv2.cvtColor(panel, cv2.COLOR_BGR2RGB))
        axes[p, c].axis("off")
plt.tight_layout()
plt.show()

## Check video

`normalized.mp4`: the original frame with the raw skeletons on the left, each fighter's normalized skeleton on the
right (the arrow is the fighter's facing on screen).

In [ ]:
if RENDER_VIDEO:
    W, H = meta["width"], meta["height"]
    small = (960, int(960 * H / W) // 2 * 2)
    panel = (480, small[1])
    rc = cfg["render"]
    with visualize.VideoWriter(out_dir / "normalized.mp4", meta["fps"], (small[0] + 2 * panel[0], small[1]), crf=rc["crf"]) as w:
        for t_, frame in tqdm(fr.iter_frames(video), total=T, desc="render"):
            left = cv2.resize(visualize.draw_frame(frame, None, keypoints[t_], t_, rc["min_joint_score"]), small)
            panels = []
            for p in range(2):
                arrow = {1: "->", -1: "<-", 0: "?"}[int(facing[t_, p])]
                ground = "ground" if result["grounded"][t_, p] else "air"
                panels.append(visualize.draw_normalized(
                    joints[t_, p], visualize.PLAYER_COLORS[p], size=panel, px_per_unit=95,
                    observed=result["observed"][t_, p], title=f"P{p + 1}  facing {arrow}  {ground}",
                ))
            w.write(np.hstack([left, *panels]))
    print("wrote", out_dir / "normalized.mp4")